# Animate data over time {#ref_tutorials_animate_time}

`MAPDL`{.interpreted-text role="bdg-mapdl"} `LS-DYNA`{.interpreted-text
role="bdg-lsdyna"} `FLUENT`{.interpreted-text role="bdg-fluent"}
`CFX`{.interpreted-text role="bdg-cfx"}

This tutorial demonstrates how to create 3D animations of data in time.

To animate data across time, the data must be stored in a
`FieldsContainer<ansys.dpf.core.fields_container.FieldsContainer>`{.interpreted-text
role="class"} with a `time` label.


# Get the result files

For this tutorial, we use a case available in the
`examples<ansys.dpf.core.examples>`{.interpreted-text role="mod"}
module. For more information about how to import your own result file in
DPF, see the `ref_tutorials_import_data`{.interpreted-text role="ref"}
tutorial section.


In [ ]:
from ansys.dpf import core as dpf
from ansys.dpf.core import examples, operators as ops

result_file_path = examples.find_msup_transient()
model = dpf.Model(data_sources=result_file_path)

# Define a time scoping

To animate across time, define the time steps of interest. This tutorial
retrieves all time steps available in
`TimeFreqSupport<ansys.dpf.core.time_freq_support.TimeFreqSupport>`{.interpreted-text
role="class"}, but you can also filter them. For more information on how
to define a scoping, see the `narrow_down_data` tutorial in the
`ref_tutorials_import_data`{.interpreted-text role="ref"} section.


In [ ]:
time_steps = model.metadata.time_freq_support.time_frequencies

# Extract the results

Extract the results to animate. In this tutorial, we extract
displacement and stress results.

:::: note
::: title
Note
:::

Only the `elemental`, `nodal`, or `faces` locations are supported for
animations. `overall` and `elemental_nodal` locations are not currently
supported.
::::

Get the displacement fields (already on nodes) at all time steps.


In [ ]:
disp_fc = model.results.displacement(time_scoping=time_steps).eval()
print(disp_fc)

Get the stress fields on nodes at all time steps. Request `nodal`
location as the default `elemental_nodal` location is not supported for
animations.


In [ ]:
stress_fc = (
    model.results.stress.on_location(location=dpf.locations.nodal)
    .on_time_scoping(time_scoping=time_steps)
    .eval()
)
print(stress_fc)

# Animate the results

Animate the results with
`FieldsContainer.animate()<ansys.dpf.core.fields_container.FieldsContainer.animate>`{.interpreted-text
role="func"}. You can animate on a deformed mesh (color map + mesh
deformation) or on a static mesh (color map only).

Default behavior of `animate()`:

- Displays the norm of the data components.
- Displays data at the top layer for shells.
- Displays the deformed mesh when animating displacements.
- Displays the static mesh for other result types.
- Uses a constant uniform scale factor of 1.0 when deforming the mesh.

You can animate any result on a deformed geometry by providing
displacement results in the `deform_by` parameter. It accepts a result
object, an
`Operator<ansys.dpf.core.dpf_operator.Operator>`{.interpreted-text
role="class"} (must evaluate to a `FieldsContainer` of the same length),
or a `FieldsContainer` of the same length.

:::: note
::: title
Note
:::

The behavior of `animate()` is defined by a
`Workflow<ansys.dpf.core.workflow.Workflow>`{.interpreted-text
role="class"} that it creates internally and feeds to an
`Animator<ansys.dpf.core.animator.Animator>`{.interpreted-text
role="class"}. This workflow loops over a field of frame indices and for
each frame generates a field of norm contours to render, as well as a
displacement field to deform the mesh if `deform_by` is provided.
::::


# Animate the displacement results --- deformed mesh


In [ ]:
disp_fc.animate()

# Animate the displacement results --- static mesh

Use `deform_by=False` to animate on a static mesh.


In [ ]:
disp_fc.animate(deform_by=False)

# Animate the stress --- deformed mesh

Pass the displacement `FieldsContainer` to `deform_by`.


In [ ]:
stress_fc.animate(deform_by=disp_fc)

Animate the stress --- static mesh


In [ ]:
stress_fc.animate()

# Change the scale factor

The scale factor can be:

- A single number for uniform constant scaling.
- A list of numbers (same length as the number of frames) for varying
  scaling.

## Uniform constant scaling


In [ ]:
uniform_scale_factor = 10.0
disp_fc.animate(scale_factor=uniform_scale_factor)

# Varying scaling


In [ ]:
varying_scale_factor = [float(i) for i in range(len(disp_fc))]
disp_fc.animate(scale_factor=varying_scale_factor)

# Save the animation

Use the `save_as` argument with a target file path. Accepted extensions
are `.gif`, `.avi`, and `.mp4`. For more information see
[pyvista.Plotter.open_movie](https://docs.pyvista.org/api/plotting/_autosummary/pyvista.Plotter.open_movie.html).


In [ ]:
stress_fc.animate(deform_by=disp_fc, save_as="animate_stress.gif")

# Control the camera

Control the camera with the `cpos` argument.

A camera position is a combination of a position, a focal point
(target), and an upwards vector:

``` python
camera_position = [
    [pos_x, pos_y, pos_z],   # position
    [fp_x, fp_y, fp_z],      # focal point
    [up_x, up_y, up_z],      # upwards vector
]
```

The `animate()` method accepts a single camera position or a list of
camera positions (one per frame).

:::: note
::: title
Note
:::

A useful technique for defining a camera position: do a first
interactive plot with `return_cpos=True`, position the camera as
desired, and retrieve the output of the plotting command.
::::

## Fixed camera


In [ ]:
cam_pos = [[0.0, 2.0, 0.6], [0.05, 0.005, 0.5], [0.0, 0.0, 1.0]]
stress_fc.animate(cpos=cam_pos)

# Moving camera


In [ ]:
import copy

cpos_list = [cam_pos]
for i in range(1, len(disp_fc)):
    new_pos = copy.deepcopy(cpos_list[i - 1])
    new_pos[0][0] += 0.1
    cpos_list.append(new_pos)

stress_fc.animate(cpos=cpos_list)

# Additional options

You can use additional PyVista arguments, such as:

- `show_axes=True` / `show_axes=False` to show or hide the coordinate
  system axis.
- `off_screen=True` to render off-screen for batch animation creation.
- `framerate` to change the frame rate.
- `quality` to change the image quality.
